# Connectome figure panels

Three headline panels, one per claim (CLAUDE.md, "Scientific objective"),
plus three domain-restricted `domain_*.png` panels — a robustness-tier check
on claim 2 (movies, videogames, stories; see CLAUDE.md, "Domain-restricted
robustness figures") placed in the same montage at the user's request.
Reads `output_data/group_stats/*.tsv` (written by `run-group-stats`) and
plots only — no connectome loading, no similarity computation, which now
lives in `analysis/group_stats.py`.

1. `longitudinal.png` — claim 1 (stable across five years): within-subject
   Pearson similarity vs. friends season lag, one line per network, against
   the between-subject floor.
2. `cross_context.png` — claim 2 (captures a variety of functional brain
   states): the four same/different-subject x same/different-dataset bins,
   all datasets, all networks.
3. `network_quality.png` — claim 3 (applies to all networks, with varying
   quality): per-network within-subject stability vs. median tSNR (falls back
   to a labelled ordering plot with a coverage note when tSNR coverage is too
   thin — see CLAUDE.md, "The QC measures asset").
4. `domain_movies.png` / `domain_videogames.png` / `domain_stories.png` — the
   same four-bin comparison as `cross_context.png`, restricted one domain at a
   time to `analysis.group_stats.DOMAIN_DATASETS`. Movies uses title-level task
   identity (friends season or movie10 title); videogames/stories reuse
   dataset-level identity, same as `cross_context.png`.

**Panels carry no legend and no title.** Both are montage-level furniture, and
inside a panel-sized canvas they crowd the data — so each panel is a bare plot
(axes, ticks, axis labels, data), and its legend is written next to it as a
standalone horizontal strip: `longitudinal_legend.png` and
`cross_context_legend.png`. Titles belong in
`output_data/connectome_figure.svg`, where they can be typeset once for the
whole multipanel figure; the coverage caveat for panel 3 is written as plain
text to `network_quality_note.txt` for the same reason. Place the legend
strips in Inkscape like any other panel — they are linked by relative path
and sized through `panel_size` exactly as the panels are.

Each is saved at exactly the size `output_data/connectome_figure.svg`
allocates it (`airoh.figures.panel_size`), at the montage's DPI, with
`layout="constrained"` and never `bbox_inches="tight"` — the two rules that
keep panel placement 1:1 (CLAUDE.md, "Figures: the Inkscape montage
pattern"). The detailed per-network histogram grids (from
`pair_histograms.tsv`) are saved alongside as extra diagnostic outputs in the
same folder, since that folder is this notebook's "already ran" sentinel.


In [1]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
import numpy as np
import pandas as pd
import yaml
from airoh.figures import panel_size

FIGURE_DPI = int(os.environ.get("FIGURE_MONTAGE_DPI", 300))

# Type sizes for the whole montage, in one place.
#
# Every panel here is placed at its true physical size (see CLAUDE.md,
# "Figures: the Inkscape montage pattern"), so a panel canvas is only ~2 in
# wide. Matplotlib's 10 pt defaults are sized for a ~6 in canvas and overflow
# one this small — at the montage's real box sizes the y-axis label was taller
# than the figure and got clipped. Scale the type down once, globally, rather
# than per-call.
plt.rcParams.update({
    "font.size": 6,
    "axes.labelsize": 6,
    "axes.titlesize": 7,
    "xtick.labelsize": 5,
    "ytick.labelsize": 5,
    "legend.fontsize": 6,
    "lines.linewidth": 1.0,
    "lines.markersize": 3,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 2.0,
    "ytick.major.size": 2.0,
})

output_dir = Path(os.environ.get("OUTPUT_DATA_DIR", "../output_data")).resolve()
figures_base = Path(os.environ.get("FIGURES_DIR", output_dir / "figures")).resolve()

project_root = output_dir.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

figure_dir = figures_base / "figure_connectomes"
figure_dir.mkdir(parents=True, exist_ok=True)

with open(project_root / "invoke.yaml") as handle:
    invoke_config = yaml.safe_load(handle)

PARCELLATION = invoke_config["parcellation"]
NETWORK_ORDER = invoke_config["parcellations"][PARCELLATION]["network_order"]
MEASURE = invoke_config.get("analysis_measure", "pearson")

group_stats_dir = output_dir / "group_stats"
cross_context = pd.read_csv(group_stats_dir / "cross_context.tsv", sep="\t")
longitudinal_bins = pd.read_csv(group_stats_dir / "longitudinal_bins.tsv", sep="\t")
longitudinal_lag = pd.read_csv(group_stats_dir / "longitudinal_lag.tsv", sep="\t")
network_quality = pd.read_csv(group_stats_dir / "network_quality.tsv", sep="\t")
session_gate = pd.read_csv(group_stats_dir / "session_gate.tsv", sep="\t")
pair_histograms = pd.read_csv(group_stats_dir / "pair_histograms.tsv", sep="\t")
domain_cross_context = pd.read_csv(group_stats_dir / "domain_cross_context.tsv", sep="\t")

print(f"📂 {PARCELLATION}, measure={MEASURE}: "
      f"{len(cross_context)} cross-context rows, {len(longitudinal_bins)} longitudinal-bin rows")


📂 cneuromod2026, measure=pearson: 180 cross-context rows, 180 longitudinal-bin rows


In [2]:
# Legends live outside the panels, as their own montage elements.
#
# A panel-sized canvas has no room for a nine-network key on top of the data,
# so each panel is drawn bare and its legend is written to a separate
# `*_legend.png`: a horizontal strip holding nothing but the key. The strip is
# a montage element like any other, so it is sized through `panel_size` and
# saved at the montage DPI with the same two rules (`layout="constrained"`,
# never `bbox_inches="tight"`).


def save_legend(handles, labels, name, default_size, ncol=None, fontsize=6):
    """Render `handles`/`labels` alone into `{name}` as a horizontal strip."""
    if not handles:
        return
    figsize = panel_size(f"figure_connectomes/{name}", default_size)
    fig = plt.figure(figsize=figsize, layout="constrained")
    fig.legend(
        handles,
        labels,
        loc="center",
        ncol=ncol or min(len(handles), 5),
        fontsize=fontsize,
        frameon=False,
    )
    fig.savefig(figure_dir / name, dpi=FIGURE_DPI)
    plt.close(fig)
    print(f"✅ wrote {figure_dir / name} at {figsize} in")


In [3]:
# Truncated y-axis for the four bar panels.
#
# No bar in any of them falls below ~0.42, so a 0-based axis spends half its
# height on empty space and squashes the within-/between-task contrast that is
# the actual result. Truncating is only honest if the reader can see it, so the
# break is marked on both vertical spines rather than left implicit.

BAR_AXIS_BOTTOM = 0.4


def mark_truncated_axis(ax, y_bottom=BAR_AXIS_BOTTOM):
    """Truncate `ax`'s y-axis at `y_bottom` and draw the break marks."""
    ax.set_ylim(y_bottom, ax.get_ylim()[1])
    break_style = dict(transform=ax.transAxes, clip_on=False, color="black",
                       linewidth=0.6, zorder=5)
    for x_position in (0, 1):
        ax.plot([x_position - 0.015, x_position + 0.015], [0.015, 0.040], **break_style)
        ax.plot([x_position - 0.015, x_position + 0.015], [0.032, 0.057], **break_style)


# One colour per network, shared by every panel that shows networks.
#
# Panel A draws one line per network, panel C one point per network, and the
# bar panels carry a colour bubble beside each x tick — so the reader learns
# the nine colours once, from panel A's legend, and can then read a network's
# identity anywhere in the montage without a second key.

# Canonical Yeo-7 literature colours, matching the `GROUP_COLORS` table in
# the cneuromod.all.qa_figures repo, so the two projects' figures name the
# same network with the same colour. Cerebellum and subcortex are not Yeo networks and get
# two off-palette hues. Yeo's own Limbic (#DCF8A4) is near-invisible on a white
# glass brain, so it is darkened here exactly as qa_figures darkens it.
NETWORK_COLORS = {
    "Vis": "#781286",          # Yeo visual (purple)
    "SomMot": "#4682B4",       # Yeo somatomotor (steel blue)
    "DorsAttn": "#00760E",     # Yeo dorsal attention (green)
    "SalVentAttn": "#C43AFA",  # Yeo ventral attention / salience (violet)
    "Limbic": "#B5B54E",       # Yeo limbic, darkened for legibility
    "Cont": "#E69422",         # Yeo frontoparietal control (orange)
    "Default": "#CD3E4E",      # Yeo default mode (red)
    "cerebellum": "#2E8B8B",   # off-palette teal (not a Yeo network)
    "subcortex": "#8B5E3C",    # off-palette brown (not a Yeo network)
}


def add_network_color_bubbles(ax, x_positions, marker_size=8):
    """Draw a network-coloured dot under each x tick of a per-network panel."""
    bubble_transform = mtransforms.blended_transform_factory(ax.transData, ax.transAxes)
    ax.scatter(
        x_positions, [-0.028] * len(x_positions),
        s=marker_size, c=[NETWORK_COLORS[network] for network in NETWORK_ORDER],
        transform=bubble_transform, clip_on=False, zorder=5,
    )
    ax.tick_params(axis="x", pad=6)


In [4]:
# Panel G — network_maps.png: the montage's network key.
#
# Nine sagittal glass brains stacked down the left edge of the page, one per
# network, each filled with that network's colour and named beside it. This is
# what makes NETWORK_COLORS readable: every other panel (A's lines, C's points,
# the bar panels' x-tick bubbles) inherits the colour, and the reader learns it
# here, anatomically, instead of from a colour swatch list. Panel A's legend
# strip therefore carries only the between-subject band.
#
# The masks come from the MNI group atlas via `analysis/atlas_maps.py` — the
# one display-only exception to "never read anat/atlases" (see that module's
# docstring). Absent atlas content just skips the panel, like every other
# credentialed asset in this project.
from analysis.atlas_maps import atlas_paths, network_mask_images

source_dir = Path(os.environ.get("SOURCE_DATA_DIR", "../source_data")).resolve()
dseg_path, labels_tsv_path = atlas_paths(source_dir / "cneuromod.all")

if dseg_path.is_file() and labels_tsv_path.is_file():
    import matplotlib as mpl
    from nilearn import plotting as nilearn_plotting

    figsize = panel_size("figure_connectomes/network_maps.png", (1.1, 5.2))
    masks = network_mask_images(dseg_path, labels_tsv_path, NETWORK_ORDER)

    # nilearn takes the axes over, so this panel keeps the plain layout engine
    # and trims its own margins — `bbox_inches="tight"` stays off everywhere.
    fig, axes = plt.subplots(len(masks), 1, figsize=figsize)
    # A strip of margin on the left holds the rotated network names; the brains
    # themselves take the rest, packed with no gap so nine tiles fit the page.
    fig.subplots_adjust(left=0.19, right=1, bottom=0.005, top=0.995, hspace=0.02)

    for ax, (network, mask_img) in zip(np.atleast_1d(axes), masks.items()):
        nilearn_plotting.plot_glass_brain(
            mask_img, axes=ax, display_mode="x", colorbar=False,
            cmap=mpl.colors.ListedColormap([NETWORK_COLORS[network]]),
            vmin=0.5, vmax=1, plot_abs=False,
        )
        ax.text(-0.04, 0.5, network, transform=ax.transAxes, rotation=90,
                ha="center", va="center", fontsize=5, clip_on=False,
                color=NETWORK_COLORS[network])

    fig.savefig(figure_dir / "network_maps.png", dpi=FIGURE_DPI)
    plt.close(fig)
    print(f"✅ wrote {figure_dir / 'network_maps.png'} at {figsize} in "
          f"({len(masks)} networks)")
else:
    print(f"⚠️  No MNI group atlas at {dseg_path} — run `invoke fetch-atlas`. "
          "Skipping network_maps.png.")


✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/figure_connectomes/network_maps.png at (1.028221535433071, 4.884051968503937) in (9 networks)


In [5]:
# Panel 1 — longitudinal.png: within-subject similarity vs. friends season lag.
#
# Broken y-axis: the per-network curves live in ~0.83-0.96 and the
# between-subject floor sits near 0.57, so a single continuous axis spends
# ~40% of the panel on empty space. The two stacked axes keep the floor in
# view — it is what the decay is measured against — without paying for the gap.
figsize = panel_size("figure_connectomes/longitudinal.png", (4.0, 4.5))

lag = longitudinal_lag[
    (longitudinal_lag["gate"] == "gated") & (longitudinal_lag["lag_type"] == "season")
]

if len(lag):
    fig, (ax_curves, ax_floor) = plt.subplots(
        2, 1, figsize=figsize, layout="constrained", sharex=True,
        height_ratios=[4, 1],
    )
    fig.get_layout_engine().set(hspace=0.04)

    curve_values = []
    for network in NETWORK_ORDER:
        within = lag[(lag["network"] == network) & (lag["pair_type"] == "within-subject")]
        within = within.sort_values("lag_value")
        if len(within):
            ax_curves.plot(within["lag_value"], within["median"], marker="o",
                           color=NETWORK_COLORS[network], label=network,
                           linewidth=1.2, markersize=3)
            curve_values.extend(within["median"].tolist())

    between = lag[(lag["pair_type"] == "between-subject")]
    if len(between):
        band = between.groupby("lag_value")["median"].median()
        ax_floor.axhspan(band.min(), band.max(), color="0.85", zorder=0,
                         label="between-subject band")
        floor_pad = max(0.01, 0.5 * (band.max() - band.min()))
        ax_floor.set_ylim(band.min() - floor_pad, band.max() + floor_pad)
    ax_floor.yaxis.set_major_locator(plt.MaxNLocator(2))

    if curve_values:
        curve_pad = 0.05 * (max(curve_values) - min(curve_values))
        ax_curves.set_ylim(min(curve_values) - curve_pad, max(curve_values) + curve_pad)

    # Hide the facing spines and mark the break on the two outer ones.
    ax_curves.spines["bottom"].set_visible(False)
    ax_curves.tick_params(bottom=False)
    ax_floor.spines["top"].set_visible(False)
    break_style = dict(clip_on=False, color="black", linewidth=0.6, zorder=5)
    for x_position in (0, 1):
        ax_curves.plot([x_position - 0.015, x_position + 0.015], [-0.012, 0.012],
                       transform=ax_curves.transAxes, **break_style)
        ax_floor.plot([x_position - 0.015, x_position + 0.015], [0.988, 1.012],
                      transform=ax_floor.transAxes, **break_style)

    ax_floor.set_xlabel("season lag")
    fig.supylabel("similarity (Fisher-z)", fontsize=6)

    # The nine network colours are keyed anatomically by network_maps.png, so
    # this strip carries only what that panel cannot show: the floor band.
    legend_handles, legend_labels = ax_floor.get_legend_handles_labels()
else:
    fig, ax_curves = plt.subplots(figsize=figsize, layout="constrained")
    ax_curves.text(0.5, 0.5, "no friends longitudinal data\n(smoke run, or single season)",
                   ha="center", va="center", transform=ax_curves.transAxes,
                   color="0.5", fontsize=6)
    ax_curves.set_xticks([])
    ax_curves.set_yticks([])
    legend_handles, legend_labels = [], []

fig.savefig(figure_dir / "longitudinal.png", dpi=FIGURE_DPI)
plt.close(fig)
print(f"✅ wrote {figure_dir / 'longitudinal.png'} at {figsize} in")

# One entry — the between-subject band. The networks are keyed by the
# glass-brain column, not repeated here.
save_legend(legend_handles, legend_labels, "longitudinal_legend.png", (2.0, 0.3), ncol=1)


✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/figure_connectomes/longitudinal.png at (1.8110236220472442, 2.047244094488189) in
✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/figure_connectomes/longitudinal_legend.png at (1.8110236220472442, 0.2362204724409449) in


In [6]:
# Panel 2 — cross_context.png: the four bins x nine networks, all datasets.
figsize = panel_size("figure_connectomes/cross_context.png", (5.0, 4.5))
fig, ax = plt.subplots(figsize=figsize, layout="constrained")

gated = cross_context[cross_context["gate"] == "gated"]
bin_order = [
    "within-subject / within-dataset", "within-subject / between-dataset",
    "between-subject / within-dataset", "between-subject / between-dataset",
]
bin_colors = {
    bin_order[0]: "#1b9e77", bin_order[1]: "#7570b3",
    bin_order[2]: "#d95f02", bin_order[3]: "#999999",
}

x = np.arange(len(NETWORK_ORDER))
width = 0.2
for i, bin_label in enumerate(bin_order):
    values = []
    for network in NETWORK_ORDER:
        row = gated[(gated["network"] == network) & (gated["bin"] == bin_label)]
        values.append(row["median"].iloc[0] if len(row) else np.nan)
    ax.bar(x + (i - 1.5) * width, values, width, color=bin_colors[bin_label],
           label=bin_label)

ax.set_xticks(x)
ax.set_xticklabels(NETWORK_ORDER, rotation=45, ha="right")
ax.set_ylabel("median similarity (Fisher-z)")
mark_truncated_axis(ax)
add_network_color_bubbles(ax, x)

legend_handles, legend_labels = ax.get_legend_handles_labels()

fig.savefig(figure_dir / "cross_context.png", dpi=FIGURE_DPI)
plt.close(fig)
print(f"✅ wrote {figure_dir / 'cross_context.png'} at {figsize} in")

# Four long labels: two columns keep the strip from running off the montage.
save_legend(legend_handles, legend_labels, "cross_context_legend.png", (5.0, 0.7), ncol=2)


✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/figure_connectomes/cross_context.png at (2.283464566929134, 2.047244094488189) in
✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/figure_connectomes/cross_context_legend.png at (4.133858267716536, 0.3937007874015748) in


In [7]:
# Panel 2b — domain_*.png: same four-bin comparison as cross_context.png,
# restricted to one naturalistic-stimulus domain at a time (CLAUDE.md,
# "Domain-restricted robustness figures") — movies (title-level task
# identity: friends season or movie10 title), videogames, stories
# (dataset-level task identity, same axis as cross_context.png).


def plot_domain_panel(domain, group_name, filename):
    """Draw one domain's four-bin x network grouped bar chart, save its legend."""
    figsize = panel_size(f"figure_connectomes/{filename}", (5.0, 4.5))
    fig, ax = plt.subplots(figsize=figsize, layout="constrained")

    subset = domain_cross_context[
        (domain_cross_context["domain"] == domain)
        & (domain_cross_context["gate"] == "gated")
    ]
    bin_order = [
        f"within-subject / within-{group_name}", f"within-subject / between-{group_name}",
        f"between-subject / within-{group_name}", f"between-subject / between-{group_name}",
    ]
    bin_colors = {
        bin_order[0]: "#1b9e77", bin_order[1]: "#7570b3",
        bin_order[2]: "#d95f02", bin_order[3]: "#999999",
    }

    x = np.arange(len(NETWORK_ORDER))
    width = 0.2
    for i, bin_label in enumerate(bin_order):
        values = []
        for network in NETWORK_ORDER:
            row = subset[(subset["network"] == network) & (subset["bin"] == bin_label)]
            values.append(row["median"].iloc[0] if len(row) else np.nan)
        ax.bar(x + (i - 1.5) * width, values, width, color=bin_colors[bin_label],
               label=bin_label)

    ax.set_xticks(x)
    ax.set_xticklabels(NETWORK_ORDER, rotation=45, ha="right")
    ax.set_ylabel("median similarity (Fisher-z)")
    mark_truncated_axis(ax)
    add_network_color_bubbles(ax, x)

    legend_handles, legend_labels = ax.get_legend_handles_labels()
    fig.savefig(figure_dir / filename, dpi=FIGURE_DPI)
    plt.close(fig)
    print(f"✅ wrote {figure_dir / filename} at {figsize} in")

    legend_name = filename.replace(".png", "_legend.png")
    save_legend(legend_handles, legend_labels, legend_name, (5.0, 0.7), ncol=2)


plot_domain_panel("movies", "title", "domain_movies.png")
plot_domain_panel("videogames", "dataset", "domain_videogames.png")
plot_domain_panel("stories", "dataset", "domain_stories.png")


✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/figure_connectomes/domain_movies.png at (1.968503937007874, 1.7322834645669292) in


✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/figure_connectomes/domain_movies_legend.png at (4.133858267716536, 0.35433070866141736) in
✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/figure_connectomes/domain_videogames.png at (1.968503937007874, 1.7322834645669292) in


✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/figure_connectomes/domain_videogames_legend.png at (5.0, 0.7) in
✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/figure_connectomes/domain_stories.png at (1.968503937007874, 1.7322834645669292) in
✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/figure_connectomes/domain_stories_legend.png at (5.0, 0.7) in


In [8]:
# Panel 3 — network_quality.png: within-subject stability vs. median tSNR per network.
# Single series, so no legend strip; the coverage caveat goes to a text file
# instead of onto the canvas, to be typeset as caption in the montage.
figsize = panel_size("figure_connectomes/network_quality.png", (3.5, 4.5))
fig, ax = plt.subplots(figsize=figsize, layout="constrained")

covered = network_quality[network_quality["n_tsnr"] > 0]
if len(covered) >= 2:
    ax.scatter(covered["median_tsnr"], covered["within_subject_median_cross_context"],
               color=[NETWORK_COLORS[network] for network in covered["network"]], s=30)
    # Nine labels on a ~45 mm panel collide unless they are placed. Try eight
    # offsets per point (right/left x four heights) against approximate text
    # boxes and keep the first that overlaps nothing already placed, falling
    # back to the least-overlapping one — enough for nine points, and it keeps
    # the panel free of an extra dependency.
    x_values = covered["median_tsnr"]
    y_values = covered["within_subject_median_cross_context"]
    x_span = x_values.max() - x_values.min()
    y_span = y_values.max() - y_values.min()
    ax.set_xlim(x_values.min() - 0.12 * x_span, x_values.max() + 0.12 * x_span)
    ax.set_ylim(y_values.min() - 0.10 * y_span, y_values.max() + 0.12 * y_span)

    # Everything below is in typographic points, the unit `offset points`
    # uses; transData returns display pixels, so convert. Draw once first so
    # constrained layout has settled the axes before we read coordinates.
    LABEL_FONTSIZE = 5
    fig.canvas.draw()
    character_width_pt = 0.60 * LABEL_FONTSIZE
    line_height_pt = 1.2 * LABEL_FONTSIZE
    offsets_pt = [(3, 1.5), (-3, 1.5),
                  (3, -line_height_pt - 1.5), (-3, -line_height_pt - 1.5),
                  (3, line_height_pt + 1.5), (-3, line_height_pt + 1.5),
                  (3, -2 * line_height_pt - 1.5), (-3, -2 * line_height_pt - 1.5)]

    placed_boxes = []

    def overlap_area(box):
        """Total area `box` shares with already-placed labels, in pt^2."""
        total = 0.0
        for other in placed_boxes:
            wide = min(box[2], other[2]) - max(box[0], other[0])
            tall = min(box[3], other[3]) - max(box[1], other[1])
            if wide > 0 and tall > 0:
                total += wide * tall
        return total

    for _, row in covered.sort_values("median_tsnr").iterrows():
        point = (row["median_tsnr"], row["within_subject_median_cross_context"])
        point_pt = ax.transData.transform(point) * 72 / fig.dpi
        text_width_pt = len(row["network"]) * character_width_pt
        best = None
        for offset_x, offset_y in offsets_pt:
            left = point_pt[0] + (offset_x if offset_x > 0 else offset_x - text_width_pt)
            box = (left, point_pt[1] + offset_y, left + text_width_pt,
                   point_pt[1] + offset_y + line_height_pt)
            area = overlap_area(box)
            if best is None or area < best[0]:
                best = (area, box, offset_x, offset_y)
            if area == 0:
                break
        _, box, offset_x, offset_y = best
        placed_boxes.append(box)
        ax.annotate(
            row["network"], point, fontsize=LABEL_FONTSIZE,
            color=NETWORK_COLORS[row["network"]],
            ha="left" if offset_x > 0 else "right",
            xytext=(offset_x, offset_y), textcoords="offset points",
        )
    ax.set_xlabel("median tSNR")
    ax.set_ylabel("within-subject similarity")
    coverage_note = ""
else:
    order = network_quality.sort_values(
        "within_subject_median_cross_context", ascending=False
    )
    ax.barh(order["network"], order["within_subject_median_cross_context"], color="#1b9e77")
    ax.invert_yaxis()
    ax.set_xlabel("within-subject similarity")
    coverage_note = (
        f"tSNR coverage too thin ({int(covered.shape[0])}/{len(network_quality)} networks) "
        "— ordering only. Expected to fill in as qa_figures' atlas_tsnr tables land upstream."
    )

fig.savefig(figure_dir / "network_quality.png", dpi=FIGURE_DPI)
plt.close(fig)
print(f"✅ wrote {figure_dir / 'network_quality.png'} at {figsize} in")

note_path = figure_dir / "network_quality_note.txt"
note_path.write_text(coverage_note + "\n" if coverage_note else "")
print(f"✅ wrote {note_path}: {coverage_note or '(no caveat — tSNR coverage sufficient)'}")


✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/figure_connectomes/network_quality.png at (1.6141732283464567, 2.047244094488189) in
✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/figure_connectomes/network_quality_note.txt: (no caveat — tSNR coverage sufficient)


In [9]:
# Diagnostic outputs (not montage panels): 3x3 density grids per analysis/measure,
# from the precomputed pair_histograms.tsv. Free-size, saved at 150 dpi.
for analysis, gate in (("cross_context", "gated"), ("longitudinal", "gated")):
    subset = pair_histograms[
        (pair_histograms["analysis"] == analysis) & (pair_histograms["gate"] == gate)
    ]
    if subset.empty:
        continue
    fig, axes = plt.subplots(3, 3, figsize=(12, 10), layout="constrained")
    fig.suptitle(f"{analysis} similarity distributions — {MEASURE}, gate={gate}")
    for network, sub_ax in zip(NETWORK_ORDER, axes.flat):
        net_hist = subset[subset["network"] == network]
        for bin_label, group in net_hist.groupby("bin"):
            group = group.sort_values("bin_left")
            centers = (group["bin_left"] + group["bin_right"]) / 2
            total = group["count"].sum()
            density = group["count"] / total if total else group["count"]
            sub_ax.plot(centers, density, label=bin_label, linewidth=1)
        sub_ax.set_title(network, fontsize=9)
        sub_ax.set_yticks([])
    axes.flat[0].legend(fontsize=6, loc="upper right")
    out_path = figure_dir / f"{analysis}_{MEASURE}_histograms.png"
    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    print(f"✅ wrote {out_path}")


✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/figure_connectomes/cross_context_pearson_histograms.png


✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/figure_connectomes/longitudinal_pearson_histograms.png
